# Kubernetes for Enterprise AI Applications (AWS + Azure)

Kubernetes (K8s) is the **de facto standard** for deploying enterprise AI, ML, and microservice applications.

For senior interviews, you should know:

- Pod
- Deployment
- Service
- Ingress
- ConfigMap
- Secret

and how they fit together.

---

# 1. What is Kubernetes?

## Definition

Kubernetes is an open-source container orchestration platform that automates the deployment, scaling, networking, self-healing, and management of containerized applications.

---

## Interview Answer

> Kubernetes is a container orchestration platform that manages Docker containers at scale. It provides automatic deployment, scaling, load balancing, self-healing, rolling updates, and service discovery for production applications.

---

# Why Kubernetes?

Imagine

```text
FastAPI

↓

Docker Container
```

Now traffic increases.

Need

```text
10 Containers
```

How will you manage

- Scaling?
- Load Balancing?
- Health Checks?
- Restart failed containers?

Kubernetes solves these problems.

---

# 2. Kubernetes Architecture

```text
                    User
                      │
                      ▼
           AWS ALB / Azure App Gateway
                      │
                      ▼
               Kubernetes Cluster
                      │
      ┌───────────────┼───────────────┐
      ▼               ▼               ▼
     Pod             Pod             Pod
  FastAPI        FastAPI        FastAPI
```

---

# Kubernetes vs Docker

| Docker | Kubernetes |
|----------|-------------|
| Creates Containers | Manages Containers |
| Single Machine | Cluster Management |
| Packaging | Orchestration |
| Build & Run | Scale & Heal |

Think of it as

```text
Docker

↓

Creates Container

↓

Kubernetes

↓

Manages Thousands of Containers
```

---

# AWS vs Azure

| AWS | Azure |
|------|--------|
| Amazon EKS | Azure Kubernetes Service (AKS) |

---

# 3. Pod

## What is a Pod?

A Pod is the **smallest deployable unit** in Kubernetes.

Usually

```
1 Pod = 1 Container
```

---

Architecture

```text
Pod

↓

FastAPI Container
```

or

```text
Pod

↓

LangGraph Container
```

---

Pod YAML

```yaml
apiVersion: v1

kind: Pod

metadata:
  name: hr-ai

spec:

  containers:

  - name: fastapi

    image: hr-ai:latest

    ports:

    - containerPort: 8000
```

---

Interview Answer

> A Pod is the smallest deployable unit in Kubernetes. It usually contains one application container along with its networking and storage configuration.

---

# 4. Deployment

Pods should never be created manually.

Deployment manages Pods.

---

Architecture

```text
Deployment

↓

Pod 1

Pod 2

Pod 3
```

---

Deployment YAML

```yaml
apiVersion: apps/v1

kind: Deployment

metadata:
  name: hr-ai

spec:

  replicas: 3

  selector:

    matchLabels:

      app: hr-ai

  template:

    metadata:

      labels:

        app: hr-ai

    spec:

      containers:

      - name: fastapi

        image: hr-ai:latest

        ports:

        - containerPort: 8000
```

---

Responsibilities

- Creates Pods
- Auto-healing
- Rolling Updates
- Scaling

---

Interview Answer

> Deployment manages Pods by maintaining the desired number of replicas, performing rolling updates, and automatically recreating failed Pods.

---

# 5. Service

Pods have dynamic IP addresses.

Need

Stable endpoint.

Service provides that.

---

Architecture

```text
Service

↓

Pod

Pod

Pod
```

---

Service YAML

```yaml
apiVersion: v1

kind: Service

metadata:

  name: hr-service

spec:

  selector:

    app: hr-ai

  ports:

  - port: 80

    targetPort: 8000

  type: ClusterIP
```

---

Service Types

| Type | Purpose |
|--------|----------|
| ClusterIP | Internal |
| NodePort | External |
| LoadBalancer | Cloud Load Balancer |

---

Interview Answer

> A Service provides a stable network endpoint and load balances traffic across multiple Pods.

---

# 6. Ingress

Users don't access Pods directly.

Traffic

↓

Ingress

↓

Service

↓

Pods

---

Architecture

```text
Internet

↓

Ingress

↓

Service

↓

Pods
```

---

Ingress YAML

```yaml
apiVersion: networking.k8s.io/v1

kind: Ingress

metadata:

  name: hr-ingress

spec:

  rules:

  - host: hr.company.com

    http:

      paths:

      - path: /

        pathType: Prefix

        backend:

          service:

            name: hr-service

            port:

              number: 80
```

---

Responsibilities

- HTTPS
- Routing
- Hostnames
- Path-based Routing

---

Interview Answer

> Ingress exposes HTTP/HTTPS services externally and routes incoming requests to Kubernetes Services based on hostnames or URL paths.

---

# 7. ConfigMap

Never hardcode

```python
DB_HOST="localhost"
```

Use ConfigMap.

---

ConfigMap YAML

```yaml
apiVersion: v1

kind: ConfigMap

metadata:

  name: hr-config

data:

  DB_HOST: postgres

  REDIS_HOST: redis
```

---

Use inside Pod

```yaml
env:

- name: DB_HOST

  valueFrom:

    configMapKeyRef:

      name: hr-config

      key: DB_HOST
```

---

Interview Answer

> ConfigMap stores non-sensitive configuration such as URLs, hostnames, and application settings separately from the application image.

---

# 8. Secret

Never store

```python
AWS_SECRET_KEY="ABC123"
```

Use Secret.

---

Secret YAML

```yaml
apiVersion: v1

kind: Secret

metadata:

  name: aws-secret

type: Opaque

stringData:

  AWS_ACCESS_KEY_ID: XXXXX

  AWS_SECRET_ACCESS_KEY: YYYYY
```

---

Use

```yaml
env:

- name: AWS_SECRET_ACCESS_KEY

  valueFrom:

    secretKeyRef:

      name: aws-secret

      key: AWS_SECRET_ACCESS_KEY
```

---

Interview Answer

> Secrets store sensitive data such as passwords, API keys, and tokens securely, keeping them separate from application code and images.

---

# 9. Complete Enterprise AI Architecture

```text
                           User
                             │
                             ▼
 AWS ALB / Azure Application Gateway
                             │
                             ▼
                        Ingress
                             │
                             ▼
                         Service
                             │
                             ▼
                Deployment (3 Replicas)
            ┌────────────┼────────────┐
            ▼            ▼            ▼
         Pod 1        Pod 2       Pod 3
      FastAPI      FastAPI      FastAPI
            │
            ▼
       LangGraph
            │
            ▼
 AWS Bedrock / Azure OpenAI
            │
            ▼
Qdrant / OpenSearch / Azure AI Search
            │
            ▼
Redis + PostgreSQL
```

---

# 10. Deployment Flow

```text
Developer

↓

GitHub

↓

GitHub Actions

↓

Docker Build

↓

Amazon ECR / Azure Container Registry

↓

Amazon EKS / Azure AKS

↓

Deployment

↓

Pods

↓

Service

↓

Ingress

↓

Users
```

---

# 11. Kubernetes Object Relationship

```text
Ingress

↓

Service

↓

Deployment

↓

Pods

↓

Containers
```

This hierarchy is frequently asked in interviews.

---

# 12. Best Practices

✅ Use Deployments instead of Pods.

✅ Store configuration in ConfigMaps.

✅ Store secrets in Secrets.

✅ Keep Pods stateless.

✅ Use health probes.

✅ Use resource requests and limits.

---

# 13. Common Interview Questions

### Q1. Pod vs Container?

Container is the application runtime.

Pod is the Kubernetes object that hosts one or more containers.

---

### Q2. Why Deployment?

To manage Pods, scaling, rolling updates, and self-healing.

---

### Q3. Why Service?

Pods have dynamic IPs.

Service provides a stable endpoint and load balancing.

---

### Q4. Why Ingress?

To expose HTTP/HTTPS services externally with routing and TLS termination.

---

### Q5. ConfigMap vs Secret?

| ConfigMap | Secret |
|------------|---------|
| Non-sensitive configuration | Sensitive configuration |
| Database Host | Database Password |
| Redis Host | API Keys |

---

### Q6. Kubernetes vs Docker?

Docker packages and runs containers.

Kubernetes manages and orchestrates containers at scale.

---

# 14. EPAM Senior Answer (3 Minutes)

> "For enterprise AI applications, I containerize the FastAPI and LangGraph application using Docker and deploy it on Amazon EKS or Azure AKS. A Deployment manages multiple Pod replicas, enabling rolling updates, self-healing, and horizontal scaling. A Kubernetes Service provides a stable endpoint and distributes traffic across Pods, while an Ingress controller exposes the application externally with HTTPS and routing rules. Non-sensitive configuration, such as database hosts or feature flags, is stored in ConfigMaps, while credentials and API keys are stored in Kubernetes Secrets or integrated with AWS Secrets Manager or Azure Key Vault. The application connects to AWS Bedrock or Azure OpenAI, Redis, PostgreSQL, and Qdrant/OpenSearch. Monitoring is handled using CloudWatch or Azure Monitor together with LangSmith for AI observability. This architecture provides a scalable, secure, and highly available deployment for enterprise AI workloads."